# Qwen3.8-27B-AWQ via vLLM

Notebook ini buat smoke test model lewat OpenAI-compatible endpoint vLLM di Cloudera AI Workbench.

Bisa dipakai untuk:
- cek endpoint vLLM
- test prompt Bahasa Indonesia / English
- ubah system prompt
- ubah temperature dan max tokens
- test mode normal vs reasoning/thinking-style


In [1]:
import os
import json
import requests

BASE_URL = os.getenv("VLLM_BASE_URL", "http://127.0.0.1:8001")
MODEL = os.getenv("MODEL_NAME", "/home/cdsw/models/Qwen3.8-27B-AWQ")

print("BASE_URL:", BASE_URL)
print("MODEL   :", MODEL)


BASE_URL: http://127.0.0.1:8001
MODEL   : /home/cdsw/models/Qwen3.8-27B-AWQ


## 1. Check vLLM server


In [2]:
r = requests.get(f"{BASE_URL}/v1/models", timeout=30)
print("HTTP:", r.status_code)
print(r.text[:3000])


HTTP: 200
{"object":"list","data":[{"id":"/home/cdsw/models/Qwen3.8-27B-AWQ","object":"model","created":1789060730,"owned_by":"vllm","root":"/home/cdsw/models/Qwen3.8-27B-AWQ","parent":null,"max_model_len":4096,"permission":[{"id":"modelperm-ab27c88e8e4d936e","object":"model_permission","created":1789060730,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}


## 2. Helper function

Set `thinking=True` untuk meminta model melakukan reasoning lebih teliti sebelum final answer.
Notebook ini tidak mencoba menampilkan hidden chain-of-thought, hanya final answer / concise reasoning summary.


In [3]:
def ask_model(
    prompt,
    system_prompt="You are a helpful business and data assistant.",
    thinking=False,
    temperature=0.2,
    max_tokens=1024,
):
    if thinking:
        system_prompt += (
            "\nReason carefully before answering. "
            "Return the final answer clearly and concisely. "
            "If useful, include only a short reasoning summary, not hidden chain-of-thought."
        )

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    response = requests.post(
        f"{BASE_URL}/v1/chat/completions",
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=300,
    )

    print("HTTP:", response.status_code)

    if response.status_code != 200:
        print(response.text[:5000])
        return None

    data = response.json()
    content = data["choices"][0]["message"]["content"]

    print("\nANSWER:\n")
    print(content)
    return data


## 3. Normal test


In [4]:
resp = ask_model(
    "Dalam 3 poin singkat, jelaskan bagaimana AI dapat membantu management menganalisis data sales.",
    thinking=False,
    temperature=0.2,
    max_tokens=512,
)


HTTP: 200

ANSWER:

Pengguna meminta dalam bahasa Indonesia: "Dalam 3 poin singkat, jelaskan bagaimana AI dapat membantu management menganalisis data sales." Saya perlu menjawab dalam 3 poin singkat. Tidak perlu alat lain. Pastikan jawaban ringkas, relevan, dan dalam bahasa Indonesia.
</think>

1. **Mendeteksi tren dan pola penjualan** – AI dapat mengidentifikasi tren, musiman, dan pola perilaku pelanggan dari data sales secara cepat.

2. **Memprediksi performa penjualan** – AI membantu memperkirakan target, potensi penjualan, dan risiko penurunan berdasarkan data historis.

3. **Memberikan rekomendasi keputusan** – AI dapat menyarankan strategi promosi, segmentasi pelanggan, atau alokasi sumber daya yang lebih efektif.


## 4. Thinking / reasoning test


In [5]:
resp = ask_model(
    '''Sebuah region mengalami sales turun 12%, inventory stockout naik dari 8% ke 21%,
sementara competitor menurunkan harga 10%.

Analisis kemungkinan penyebab utama, urutkan berdasarkan prioritas,
dan berikan rekomendasi action untuk management.''',
    thinking=True,
    temperature=0.1,
    max_tokens=1200,
)


HTTP: 200

ANSWER:

Kita perlu menjawab dalam bahasa Indonesia. User meminta analisis kemungkinan penyebab utama, urutan prioritas, dan rekomendasi action untuk management. Data: region mengalami sales turun 12%, inventory stockout naik dari 8% ke 21%, competitor menurunkan harga 10%. Perlu analisis bisnis. Kita perlu memberikan jawaban terstruktur. Tidak perlu data tambahan, tapi bisa asumsi. Harus final saja.

Kita perlu memikirkan kemungkinan penyebab utama:
1. Stockout meningkat drastis 8% ke 21% -> kehilangan penjualan langsung karena produk tidak tersedia. Ini kemungkinan penyebab utama penurunan sales 12%, terutama jika item yang stockout adalah high-velocity.
2. Competitor price cut 10% -> tekanan harga, kehilangan share ke kompetitor, terutama jika produk substitusi dan sensitivitas harga tinggi. Bisa memperburuk penurunan.
3. Demand shift/seasonality/market conditions? Mungkin faktor eksternal, tapi data tidak menyebut. Bisa jadi penyebab sekunder.
4. Supply chain/procurement

## 5. Test Bahasa Indonesia


In [6]:
ask_model(
    "Produk mana yang perlu diprioritaskan jika sales turun tetapi demand tetap tinggi dan stockout meningkat?",
    thinking=True,
    temperature=0.1,
    max_tokens=1000,
)


HTTP: 200

ANSWER:

We need answer in Indonesian. User asks: "Produk mana yang perlu diprioritaskan jika sales turun tetapi demand tetap tinggi dan stockout meningkat?" Means: Which products need to be prioritized if sales drop but demand remains high and stockout increases? Need likely business/inventory prioritization. Need answer concise but with reasoning summary. Need maybe identify products with high demand, high stockout, sales decline due to unavailability, high margin/strategic, etc. Could propose prioritization criteria: high demand index, stockout rate, lost sales, margin, lead time, customer impact. Maybe answer: prioritize products with high demand + high stockout + sales decline, especially high margin/strategic, long lead time, substitute availability. Need maybe mention use data: rank by lost sales = demand - sales, stockout rate, margin. Could include formula. Need final in Indonesian. Ensure no hidden chain-of-thought. Provide actionable.
</think>

Prioritaskan produk

{'id': 'chatcmpl-b7681c59b0e7615e',
 'object': 'chat.completion',
 'created': 1789060885,
 'model': '/home/cdsw/models/Qwen3.8-27B-AWQ',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'We need answer in Indonesian. User asks: "Produk mana yang perlu diprioritaskan jika sales turun tetapi demand tetap tinggi dan stockout meningkat?" Means: Which products need to be prioritized if sales drop but demand remains high and stockout increases? Need likely business/inventory prioritization. Need answer concise but with reasoning summary. Need maybe identify products with high demand, high stockout, sales decline due to unavailability, high margin/strategic, etc. Could propose prioritization criteria: high demand index, stockout rate, lost sales, margin, lead time, customer impact. Maybe answer: prioritize products with high demand + high stockout + sales decline, especially high margin/strategic, long lead time, substitute availability. Need maybe mention use data

## 6. Test English


In [7]:
ask_model(
    "What are the key factors management should investigate when regional sales decline while inventory availability also drops?",
    thinking=True,
    temperature=0.1,
    max_tokens=1000,
)


HTTP: 200

ANSWER:

We need answer user: "What are the key factors management should investigate when regional sales decline while inventory availability also drops?" Need provide business analysis. Need final concise but thorough. Let's think.

Scenario: regional sales decline and inventory availability drops. Need key factors management should investigate. Could be demand, supply, pricing, competition, channel, data, seasonality, macro, product mix, customer behavior, logistics, stockouts, allocation, returns, promotions, inventory accuracy, supplier issues, quality, service levels, marketing, economic conditions, weather, local events, channel shifts, e-commerce, inventory availability definition (sellable inventory, on-hand, in-transit, allocated). Need structure: verify data, demand-side, supply-side, market/competitive, operational, financial/pricing, customer/channel, external. Also note correlation: sales decline may cause inventory drops if sell-through, but if availability dr

{'id': 'chatcmpl-b6653480ba25bbf1',
 'object': 'chat.completion',
 'created': 1789060928,
 'model': '/home/cdsw/models/Qwen3.8-27B-AWQ',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'We need answer user: "What are the key factors management should investigate when regional sales decline while inventory availability also drops?" Need provide business analysis. Need final concise but thorough. Let\'s think.\n\nScenario: regional sales decline and inventory availability drops. Need key factors management should investigate. Could be demand, supply, pricing, competition, channel, data, seasonality, macro, product mix, customer behavior, logistics, stockouts, allocation, returns, promotions, inventory accuracy, supplier issues, quality, service levels, marketing, economic conditions, weather, local events, channel shifts, e-commerce, inventory availability definition (sellable inventory, on-hand, in-transit, allocated). Need structure: verify data, demand-sid

## 7. Custom Playground

Edit bagian ini sesuka Lu.


In [8]:
SYSTEM_PROMPT = '''You are Tempo Scan Commercial Intelligence Assistant.
Focus on sales, product, outlet, customer, inventory, market signals, and executive insights.
Answer concisely and clearly in the language used by the user.
Do not invent unavailable business facts.'''

USER_PROMPT = '''Kenapa sales Product A turun di Jawa Barat bulan ini?
Asumsikan inventory availability turun dan competitor promotion lebih agresif.'''

THINKING = True
TEMPERATURE = 0.2
MAX_TOKENS = 1200

resp = ask_model(
    USER_PROMPT,
    system_prompt=SYSTEM_PROMPT,
    thinking=THINKING,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)


HTTP: 200

ANSWER:

We need answer in Indonesian. User asks: "Kenapa sales Product A turun di Jawa Barat bulan ini? Asumsikan inventory availability turun dan competitor promotion lebih agresif." Need concise, clear. We are Tempo Scan Commercial Intelligence Assistant. Need not invent unavailable business facts. They ask to assume inventory availability turun and competitor promotion lebih agresif. So answer likely: sales turun karena kombinasi stok terbatas dan promo kompetitor. Need maybe structure: penyebab utama, dampak, rekomendasi. Ensure no hidden chain of thought. Final in Indonesian.
</think>

Penurunan sales Product A di Jawa Barat bulan ini kemungkinan besar disebabkan oleh dua faktor utama:

1. **Inventory availability turun**  
   Stok Product A tidak tersedia cukup di outlet, sehingga permintaan tidak dapat dipenuhi dan penjualan langsung berkurang.

2. **Promosi kompetitor lebih agresif**  
   Kompetitor sedang melakukan promosi yang lebih kuat, sehingga konsumen lebih t

## 8. Inspect raw response


In [9]:
if resp:
    print(json.dumps(resp, indent=2, ensure_ascii=False)[:10000])


{
  "id": "chatcmpl-a8af7ab65193f358",
  "object": "chat.completion",
  "created": 1789060967,
  "model": "/home/cdsw/models/Qwen3.8-27B-AWQ",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "We need answer in Indonesian. User asks: \"Kenapa sales Product A turun di Jawa Barat bulan ini? Asumsikan inventory availability turun dan competitor promotion lebih agresif.\" Need concise, clear. We are Tempo Scan Commercial Intelligence Assistant. Need not invent unavailable business facts. They ask to assume inventory availability turun and competitor promotion lebih agresif. So answer likely: sales turun karena kombinasi stok terbatas dan promo kompetitor. Need maybe structure: penyebab utama, dampak, rekomendasi. Ensure no hidden chain of thought. Final in Indonesian.\n</think>\n\nPenurunan sales Product A di Jawa Barat bulan ini kemungkinan besar disebabkan oleh dua faktor utama:\n\n1. **Inventory availability turun**  \n   Stok Pro